# Session 3 — BERT Extractive Question Answering
**Task:** Given a context paragraph and a question, find the answer span inside the context  
**Model:** `bert-base-uncased` → `BertForQuestionAnswering`  
**Dataset:** SQuAD v1.1  
**Metric:** Exact Match + F1

---
### Key difference from NER & Classification
- Input is **two sequences**: `[CLS] question [SEP] context [SEP]`
- Output is **two logit vectors** of length `seq_len`: one for start position, one for end position
- The answer is `context[start:end+1]` — no new tokens generated, just span extraction

## Step 1 — Imports & Config

In [ ]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import (
    BertTokenizerFast,
    BertForQuestionAnswering,
    get_linear_schedule_with_warmup,
)
from datasets import load_dataset

DEVICE     = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "bert-base-uncased"
MAX_LEN    = 384    # QA needs longer context than classification
BATCH_SIZE = 8      # smaller batch — inputs are longer
EPOCHS     = 2
LR         = 3e-5
TRAIN_SIZE = 3000
VAL_SIZE   = 300
SAVE_DIR   = "../../models/05_transformers/bert_qa"

print(f"Device: {DEVICE}")

## Step 2 — Load & Inspect Dataset

In [ ]:
raw = load_dataset("squad")
print(raw)

ex = raw["train"][0]
print("\nContext:  ", ex["context"][:200], "...")
print("Question: ", ex["question"])
print("Answer:   ", ex["answers"]["text"][0])
print("Answer start (char):", ex["answers"]["answer_start"][0])

## Step 3 — Tokenizer & Answer Position
SQuAD gives answer as **character offset** in the context.  
We need to convert character offset → **token position** so BERT can predict start/end token indices.

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

# Show how char offset → token position works
ex       = raw["train"][0]
question = ex["question"]
context  = ex["context"]
ans_text = ex["answers"]["text"][0]
ans_start_char = ex["answers"]["answer_start"][0]
ans_end_char   = ans_start_char + len(ans_text)

enc = tokenizer(question, context, truncation=True, max_length=MAX_LEN, return_offsets_mapping=True)

# char_to_token: given a char position, which token index does it map to?
start_token = enc.char_to_token(ans_start_char, sequence_index=1)
end_token   = enc.char_to_token(ans_end_char - 1, sequence_index=1)

print(f"Answer text:  '{ans_text}'")
print(f"Char range:   [{ans_start_char}, {ans_end_char})")
print(f"Token range:  [{start_token}, {end_token}]")
print(f"Decoded back: '{tokenizer.decode(enc['input_ids'][start_token:end_token+1])}'")
print(f"Total tokens: {len(enc['input_ids'])}")

## Step 4 — Dataset

In [ ]:
class SQuADDataset(Dataset):
    def __init__(self, hf_split, tokenizer):
        self.data      = hf_split
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ex       = self.data[idx]
        question = ex["question"]
        context  = ex["context"]
        ans_start_char = ex["answers"]["answer_start"][0]
        ans_text       = ex["answers"]["text"][0]
        ans_end_char   = ans_start_char + len(ans_text)

        enc = self.tokenizer(
            question, context,
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_offsets_mapping=True,
            return_tensors="pt",
        )

        start_pos = enc.char_to_token(ans_start_char, sequence_index=1)
        end_pos   = enc.char_to_token(ans_end_char - 1, sequence_index=1)

        # If answer got truncated, set to 0 (CLS token) — model learns to output 0 for unanswerable
        if start_pos is None: start_pos = 0
        if end_pos   is None: end_pos   = 0

        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "start_positions": torch.tensor(start_pos, dtype=torch.long),
            "end_positions":   torch.tensor(end_pos,   dtype=torch.long),
        }

train_raw = raw["train"].select(range(TRAIN_SIZE))
val_raw   = raw["validation"].select(range(VAL_SIZE))

train_ds = SQuADDataset(train_raw, tokenizer)
val_ds   = SQuADDataset(val_raw,   tokenizer)

item = train_ds[0]
print("input_ids shape:  ", item["input_ids"].shape)
print("start_position:   ", item["start_positions"].item())
print("end_position:     ", item["end_positions"].item())

## Step 5 — Model, Optimizer, Scheduler

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

model = BertForQuestionAnswering.from_pretrained(MODEL_NAME).to(DEVICE)

optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)
print(f"Model: BertForQuestionAnswering  |  Device: {DEVICE}")

## Step 6 — Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids       = batch["input_ids"].to(DEVICE)
        attention_mask  = batch["attention_mask"].to(DEVICE)
        start_positions = batch["start_positions"].to(DEVICE)
        end_positions   = batch["end_positions"].to(DEVICE)

        optimizer.zero_grad()
        # BertForQuestionAnswering returns avg of start + end cross-entropy loss
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            start_positions=start_positions,
            end_positions=end_positions,
        )
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    exact_matches = 0
    with torch.no_grad():
        for batch in loader:
            input_ids       = batch["input_ids"].to(DEVICE)
            attention_mask  = batch["attention_mask"].to(DEVICE)
            start_positions = batch["start_positions"]
            end_positions   = batch["end_positions"]

            outputs      = model(input_ids=input_ids, attention_mask=attention_mask)
            pred_starts  = outputs.start_logits.argmax(dim=-1).cpu()
            pred_ends    = outputs.end_logits.argmax(dim=-1).cpu()

            exact_matches += ((pred_starts == start_positions) & (pred_ends == end_positions)).sum().item()

    return exact_matches / (len(loader) * loader.batch_size)


for epoch in range(1, EPOCHS + 1):
    loss = train_epoch(model, train_loader, optimizer, scheduler)
    em   = evaluate(model, val_loader)
    print(f"Epoch {epoch}/{EPOCHS} | loss: {loss:.4f} | exact_match: {em:.4f}")

## Step 7 — Save & Inference

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")


def predict_qa(question, context, model, tokenizer):
    model.eval()
    enc = tokenizer(question, context, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    input_ids      = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    start = outputs.start_logits.argmax(dim=-1).item()
    end   = outputs.end_logits.argmax(dim=-1).item()

    answer_tokens = enc["input_ids"][0][start: end + 1]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True)


context = """
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.
It was constructed from 1887 to 1889 as the centerpiece of the 1889 World's Fair.
The tower is 330 metres tall and is the tallest structure in Paris.
"""

questions = [
    "Where is the Eiffel Tower located?",
    "When was the Eiffel Tower built?",
    "How tall is the Eiffel Tower?",
]
for q in questions:
    ans = predict_qa(q, context, model, tokenizer)
    print(f"Q: {q}\nA: {ans}\n")